# Transforming Circuits Data


1  Keept analytics specific columns

2  Standardized column names using snake_case

3	Renamed columns

4	Quality checks    


    -Removed duplicate records
    -Removed null records


In [0]:
from pyspark.sql import functions as F

In [0]:
%run "../00-Common/01. Env Config"

In [0]:
bronze_table=f"{catalog_name}.{bronze_schema}.circuits"
silver_table=f"{catalog_name}.{silver_schema}.circuits"

In [0]:
# Reading the table from Bronze Layer
#circuits_df=spark.read.option('versionAsof',0).table(bronze_table)
circuits_df=spark.read.table(bronze_table)



In [0]:
# Keeping only required columns

circuits_selected_df=circuits_df.select(
    F.col('circuitsId'),
    F.col('circuitName'),
    F.col('lat'),
    F.col('long'),
    F.col('locality'),
    F.col('country').alias('country_name'),
    F.col('ingestion_Timestamp'),
    F.col('source_file'))

In [0]:
# Renaming columns

circuits_renamed_df=(
                circuits_selected_df.withColumnsRenamed({
                            'circuitsId':'circuit_id',
                            'circuitName':'circuit_name',
                            'lat':'latitude',
                            'long':'longitude'
                            #{('locality','city'}
                }))

In [0]:
# Filtering out null records
circuits_filtered_df=circuits_renamed_df.filter(F.col("circuit_id").isNotNull())
 

In [0]:
# Removing duplicates if any


circuits_filtered_df=circuits_filtered_df.dropDuplicates()
display(circuits_filtered_df)

In [0]:
circuits_final_df=(circuits_filtered_df
                                      .withColumn("circuit_name",F.initcap(F.col("circuit_name")))    
                                      .withColumn("locality",F.initcap(F.col("locality"))))

In [0]:
(
    circuits_final_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(silver_table)

)